In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os, time, random, json
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
NUM_CLASSES  = 62   # 0-9, A-Z, a-z
IMG          = 28
BS           = 64
EPOCHS       = 50 #60
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.03
DROP_PATH    = 0.10
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results"
AUTOTUNE     = tf.data.AUTOTUNE

os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  EMNIST BYCLASS via tensorflow_datasets
#  label mapping (already 0-indexed): 0-9 digits, 10-35 A-Z, 36-61 a-z
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Loading EMNIST ByClass via tensorflow_datasets ...")

train_raw, info = tfds.load(
    "emnist/byclass",
    split="train",
    as_supervised=True,
    with_info=True,
    shuffle_files=True,
)
test_raw = tfds.load("emnist/byclass", split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG, IMG])
    image = image / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

# def augment(image, label):
#     image = tf.image.random_brightness(image, 0.20)
#     image = tf.image.random_contrast(image, 0.80, 1.20)
#     image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
#     image = tf.image.random_crop(image, [IMG, IMG, 1])
#     ## image = tf.image.random_flip_left_right(image)
#     image = image + tf.random.normal(tf.shape(image), stddev=0.02)
#     image = tf.clip_by_value(image, -1.0, 1.0)
#     return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.20)
    image = tf.image.random_contrast(image, 0.80, 1.20)
    image = tf.pad(image, [[3, 3], [3, 3], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    # NO horizontal flip — it breaks letter/digit identity
    # Add rotation ±10 degrees using affine transform
    angle  = tf.random.uniform([], -0.175, 0.175)
    cos_a  = tf.cos(angle)
    sin_a  = tf.sin(angle)
    transform = [cos_a, -sin_a, 0.0,
                 sin_a,  cos_a, 0.0,
                 0.0,    0.0]
    image = tf.raw_ops.ImageProjectiveTransformV3(
        images=image[tf.newaxis],
        transforms=[transform],
        output_shape=tf.shape(image)[:2],
        interpolation="BILINEAR",
        fill_mode="CONSTANT",
        fill_value=-1.0
    )[0]
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

# def augment(image, label):
#     image = tf.image.random_brightness(image, 0.20)
#     image = tf.image.random_contrast(image, 0.80, 1.20)
#     image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
#     image = tf.image.random_crop(image, [IMG, IMG, 1])
    
#     # Add rotation using tf.image.rot90() combinations or affine transform
#     angle = tf.random.uniform([], -0.15, 0.15)  # ±~9 degrees
    
#     # Convert angle to radians and create rotation matrix
#     angle_rad = angle * (3.14159265359 / 180.0)
#     cos_angle = tf.cos(angle_rad)
#     sin_angle = tf.sin(angle_rad)
    
#     # 2x3 affine transformation matrix for rotation around center
#     h, w = IMG, IMG
#     transform = [
#         [cos_angle, -sin_angle, sin_angle * (h/2 - 0.5) + cos_angle * (w/2 - 0.5)],
#         [sin_angle,  cos_angle, -sin_angle * (w/2 - 0.5) + cos_angle * (h/2 - 0.5)]
#     ]
    
#     image = tf.image.transform(image, transform, interpolation='BILINEAR')
    
#     image = image + tf.random.normal(tf.shape(image), stddev=0.02)
#     image = tf.clip_by_value(image, -1.0, 1.0)
#     return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(20_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

def apply_mixup(images, labels, alpha=0.3):
    bs      = tf.shape(images)[0]
    lam     = tf.cast(
        tf.random.uniform([bs], 0.0, 1.0), tf.float32
    )
    lam     = tf.maximum(lam, 1.0 - lam)   # keep dominant class
    idx     = tf.random.shuffle(tf.range(bs))
    lam_i   = lam[:, tf.newaxis, tf.newaxis, tf.newaxis]
    lam_l   = lam[:, tf.newaxis]
    mixed_images = lam_i * images + (1.0 - lam_i) * tf.gather(images, idx)
    mixed_labels = lam_l * labels + (1.0 - lam_l) * tf.gather(labels, idx)
    return mixed_images, mixed_labels

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .map(lambda x, y: apply_mixup(x, y, alpha=0.3))
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (                          # integer labels for F1 / per-class acc
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (                       # one-hot for model.evaluate
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

2026-04-13 08:39:11.760823: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776069551.976379      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776069552.040380      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776069552.570933      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776069552.570978      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776069552.570983      55 computation_placer.cc:177] computation placer alr

[INFO] Loading EMNIST ByClass via tensorflow_datasets ...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

...

I0000 00:00:1776069594.516797      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776069594.522829      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Shuffling /root/tensorflow_datasets/emnist/byclass/incomplete.HBRPOI_3.1.0/emnist-train.tfrecord*...:   0%|   …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/byclass/incomplete.HBRPOI_3.1.0/emnist-test.tfrecord*...:   0%|    …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/byclass/3.1.0. Subsequent calls will reuse this data.
[INFO] Train: 628,139 | Val: 69,793 | Test: 116,323
[INFO] Steps/epoch: 9814


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────

def gelu(x):
    return tf.nn.gelu(x)


class DropPath(keras.layers.Layer):
    def __init__(self, drop_prob=0.0, **kwargs):
        super().__init__(**kwargs)
        self.drop_prob = drop_prob

    def call(self, x, training=None):
        if not training or self.drop_prob == 0.0:
            return x
        keep  = 1.0 - self.drop_prob
        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape) - 1)
        mask  = tf.floor(keep + tf.random.uniform(shape, dtype=x.dtype))
        return x * mask / keep

    def get_config(self):
        cfg = super().get_config()
        cfg["drop_prob"] = self.drop_prob
        return cfg


class GatedScale(keras.layers.Layer):
    def call(self, inputs):
        features, gate = inputs
        return features * gate[:, 0:1]

    def get_config(self):
        return super().get_config()


def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])


def residual_block(x, channels, drop_rate=0.0):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if drop_rate > 0:
        x = DropPath(drop_rate)(x)
    x = layers.Add()([x, r])
    x = layers.Activation(gelu)(x)
    return x


def dense_res_block(x, in_ch, out_ch, drop_rate=0.0):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch, drop_rate)
    r2  = residual_block(r1, out_ch, drop_rate)
    r3  = residual_block(r2, out_ch, drop_rate)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out


def cross_scale_transformer_bridge(s1, s2, s3, dim=256, num_heads=4):
    def pool_proj(feat):
        v = layers.GlobalAveragePooling2D()(feat)
        v = layers.Dense(dim, activation=gelu)(v)
        return v[:, tf.newaxis, :]
    seq = layers.Concatenate(axis=1)([pool_proj(s1), pool_proj(s2), pool_proj(s3)])
    att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    att = layers.LayerNormalization()(att + seq)
    out = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(att)
    return layers.Dense(dim, activation=gelu)(out)


def adaptive_filter_capsule(x, num_classes, capsule_dim=16):
    h   = layers.Dense(256, activation=gelu)(x)
    h   = layers.Dense(num_classes * capsule_dim)(h)
    h   = layers.Reshape((num_classes, capsule_dim))(h)
    xe  = layers.RepeatVector(num_classes)(x)
    xs  = layers.Lambda(lambda t: t[:, :, :capsule_dim])(xe)
    cap = layers.Multiply()([xs, h])
    cap = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(cap)
    return layers.BatchNormalization()(cap)


def english_topology_module(x, out_features):
    # Symmetric kernels + dilated convs for Latin strokes (curves, ascenders, descenders)
    h  = layers.Conv2D(48, (1, 3), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(48, (3, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    d4 = layers.Conv2D(32, 3, padding="same", dilation_rate=4, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2, d4])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    return layers.Dense(out_features, activation=gelu)(out)


# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────

# def build_model(num_classes=62, image_size=64, drop_path_rate=0.10):
#     K   = num_classes
#     DR  = drop_path_rate
#     inp = Input(shape=(image_size, image_size, 1), name="input")

#     # 3-path English stem: curves (3×3), horizontals (1×5), verticals (5×1)
#     # English has strong vertical strokes (l, i, t, f) and curves (c, o, e)
#     t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
#     t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)

#     h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
#     h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)

#     v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
#     v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

#     stem = layers.Concatenate()([t, h, v])          # 64ch
#     stem = channel_attention(stem, 64)
#     stem = layers.Conv2D(64, 1, use_bias=False)(stem)
#     stem = layers.BatchNormalization()(stem)
#     stem = layers.Activation(gelu)(stem)

#     enc1 = dense_res_block(stem, 64,  64,  drop_rate=DR * 0.5)
#     enc2 = dense_res_block(enc1, 64,  128, drop_rate=DR)
#     enc3 = dense_res_block(enc2, 128, 256, drop_rate=DR)

#     cstb    = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
#     gap     = layers.GlobalAveragePooling2D()(enc3)
#     afc_in  = layers.Add()([gap, cstb])
#     afc_out = adaptive_filter_capsule(afc_in, K)

#     stm_out  = english_topology_module(enc3, 256)
#     combined = layers.Concatenate()([stm_out, afc_out])
#     gate     = layers.Dense(2, activation="softmax", name="gate")(combined)
#     scaled   = GatedScale()([stm_out, gate])

#     fused = layers.Concatenate()([scaled, afc_out])
#     x     = layers.Dense(512)(fused)
#     x     = layers.LayerNormalization()(x)
#     x     = layers.Activation(gelu)(x)
#     out   = layers.Dense(K, name="logits")(x)

#     return Model(inputs=inp, outputs=out, name="EnglishNet")


def build_model(num_classes=62, image_size=28, drop_path_rate=0.10):
    K   = num_classes
    DR  = drop_path_rate
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem: 3-path, balanced horizontal + vertical ──────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t)
    t = layers.Activation(gelu)(t)

    h = layers.Conv2D(16, (1, 3), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Activation(gelu)(h)

    v = layers.Conv2D(16, (3, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v)
    v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])          # 64ch
    stem = channel_attention(stem, 64)              # KEEP — cheap, always helps
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder: identical backbone — no changes needed ───────────────────
    enc1 = dense_res_block(stem, 64,  64,  drop_rate=DR * 0.5)
    enc2 = dense_res_block(enc1, 64,  128, drop_rate=DR)
    enc3 = dense_res_block(enc2, 128, 256, drop_rate=DR)

    # ── Multi-scale fusion: simple GAP concat replaces CSTB ───────────────
    # (or keep CSTB — it's cheap and doesn't hurt)
    # gap1 = layers.Dense(128, activation=gelu)(
    #            layers.GlobalAveragePooling2D()(enc1))
    # gap2 = layers.Dense(128, activation=gelu)(
    #            layers.GlobalAveragePooling2D()(enc2))
    # gap3 = layers.GlobalAveragePooling2D()(enc3)       # already 256-dim

    # multi_scale = layers.Concatenate()([gap1, gap2, gap3])  # 512-dim
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    gap3     = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([gap3, cstb_out])


    # ── Topology module: KEEP — genuinely useful for Latin strokes ─────────
    stm_out = english_topology_module(enc3, 256)

    # ── Simple fusion: no gate, no capsules ────────────────────────────────
    # fused = layers.Concatenate()([stm_out, multi_scale])    # 768-dim
    fused = layers.Concatenate()([stm_out, afc_in])    # 768-dim

    # ── Classifier: properly sized, with dropout ───────────────────────────
    x   = layers.Dense(256, use_bias=False)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    x   = layers.Dropout(0.25)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="EnglishNet")


# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────

class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}


# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────

model        = build_model(NUM_CLASSES, IMG, DROP_PATH)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)

print(f"[INFO] Params: {model.count_params():,}")

# Count class frequencies and pass class_weight to model.fit()
class_counts = np.bincount([label for _, label in tfds.as_numpy(train_raw)], minlength=62)
max_count = float(class_counts.max())
# class_weights = {i: float(class_counts.max()) / float(c + 1) for i, c in enumerate(class_counts)}
# class_weights = {i: max_count / count for i, count in enumerate(class_counts)}
class_weights = {
    i: float(np.sqrt(max_count / float(max(c, 1))))
    for i, c in enumerate(class_counts)
}
# history = model.fit(train_ds, ..., class_weight=class_weights)


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[
        ModelCheckpoint(
            os.path.join(RESULTS_DIR, "best_model.keras"),
            monitor="val_accuracy", save_best_only=True, verbose=0,
        ),
        EarlyStopping(
            monitor="val_accuracy", patience=15, #12
            restore_best_weights=True, verbose=1,
        ),
    ],
)

[INFO] Params: 5,994,446
Epoch 1/50
9815/9815 ━━━━━━━━━━━━━━━━━━━━ 840s 80ms/step - accuracy: 0.3033 - loss: 5.5784 - val_accuracy: 0.8371 - val_loss: 0.7610
Epoch 2/50
9815/9815 ━━━━━━━━━━━━━━━━━━━━ 761s 77ms/step - accuracy: 0.7157 - loss: 2.9740 - val_accuracy: 0.8452 - val_loss: 0.6823
Epoch 3/50
8793/9815 ━━━━━━━━━━━━━━━━━━━━ 1:17 76ms/step - accuracy: 0.7169 - loss: 2.5295

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────

test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

LABELS = (
    [str(d) for d in range(10)]
    + [chr(c) for c in range(65, 91)]
    + [chr(c) for c in range(97, 123)]
)

tp = np.zeros(NUM_CLASSES)
fp = np.zeros(NUM_CLASSES)
fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES)
total_per_class   = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

# ─────────────────────────────────────────────────────────────────────────────
#  SAVE
# ─────────────────────────────────────────────────────────────────────────────

results = {
    "dataset":       "EMNIST_BYCLASS",
    "num_classes":   NUM_CLASSES,
    "test_acc":      round(test_acc, 2),
    "macro_f1":      round(macro_f1, 2),
    "test_loss":     round(float(test_loss), 4),
    "params":        model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}

with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)

print(f"\n[INFO] Saved to {RESULTS_DIR}/")
print("[DONE]")